# 09b_ExtractResNet50 — Trích thêm đặc trưng ResNet50 để so với ResNet18 (Task 7a, bổ sung)

Không sửa `09_ImageFeatures.ipynb`. Notebook này dùng lại `img_feats/video_map.csv` (không ghép video lại) và trích đặc trưng bằng **ResNet50** vào thư mục riêng `Data_321/img_feats_r50/`, tên file giống hệt bản ResNet18.

**Công bằng khi so sánh:** cùng 4 phiên bản mỗi video và **cùng hạt giống ngẫu nhiên** cho augmentation (`seed = 1000 + i`), nên hai bộ đặc trưng nhìn đúng cùng những hình ảnh; chỉ khác CNN. ResNet50 cho **2048 số mỗi khung** (ResNet18 là 512), file lớn hơn khoảng 4 lần.

**Cách chạy:** kích hoạt môi trường `C:\envs\train-gpu\Scripts\activate.bat`, mở notebook bằng kernel `train-gpu`, chạy lần lượt cell 1 → 5. Lần đầu chạy cell 2 tải trọng số ResNet50 nên cần internet. Chạy lại sẽ bỏ qua video đã có file.

In [1]:
#cell 1: thư viện, kiểm tra CUDA và các file cần có, đọc lại video_map.csv của bước 09
import time
import numpy as np, pandas as pd, torch, cv2
import torch.nn as nn, torch.nn.functional as F
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook này không chạy trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
SRC_DIR = BASE / "img_feats"                       # đặc trưng ResNet18 từ 09
OUT_DIR = BASE / "img_feats_r50"                   # đặc trưng ResNet50 sẽ lưu ở đây
OUT_DIR.mkdir(exist_ok=True)
assert (SRC_DIR / "video_map.csv").exists(), "Chưa có video_map.csv: chạy 09_ImageFeatures.ipynb trước."
vm = pd.read_csv(SRC_DIR / "video_map.csv")
assert all(Path(p).exists() for p in vm.video), "Có video gốc không còn tồn tại, kiểm tra lại thư mục dữ liệu."
print("GPU:", torch.cuda.get_device_name(0), "| số video:", len(vm), "| thư mục lưu:", OUT_DIR)

GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU | số video: 215 | thư mục lưu: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\img_feats_r50


In [2]:
#cell 2: ResNet50 đã học sẵn (bỏ lớp phân loại) làm bộ trích đặc trưng, chạy CUDA fp16
import torchvision

backbone = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
backbone.fc = nn.Identity()
backbone = backbone.to(dev).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    print("Số đặc trưng mỗi khung:", backbone(torch.zeros(2, 3, 224, 224, device=dev)).shape[1])

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\Acer Nitro 5 Tiger/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100.0%


Số đặc trưng mỗi khung: 2048


In [3]:
#cell 3: đọc video, tạo 4 phiên bản (giống hệt 09), trích đặc trưng; thử video đầu và đối chiếu số khung với bản ResNet18
CHUNK = 64                                                    # lô nhỏ hơn 09 vì ResNet50 tốn VRAM hơn (GPU 4 GB)

def read_frames(path):
    cap = cv2.VideoCapture(str(path)); frames = []
    while True:
        ok, f = cap.read()
        if not ok:
            break
        frames.append(cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2RGB), (224, 224), interpolation=cv2.INTER_AREA))
    cap.release()
    return np.stack(frames)                                   # (T,224,224,3) uint8

def make_version(x, flip, s, dx, dy, b, c):
    """x: (n,3,224,224) float 0-1 trên GPU. Mọi khung của một video dùng chung một phép biến đổi."""
    if s < 1:
        k = int(224 * s); ox, oy = int(dx * (224 - k)), int(dy * (224 - k))
        x = F.interpolate(x[:, :, oy:oy + k, ox:ox + k], size=224, mode="bilinear", align_corners=False)
    x = ((x - 0.5) * c + 0.5) * b
    return (x.flip(-1) if flip else x).clamp(0, 1)

VERSIONS = [(False, False), (True, False), (False, True), (True, True)]   # (lật gương, augmentation)

@torch.no_grad()
def extract(path, seed):
    x = torch.from_numpy(read_frames(path)).to(dev).permute(0, 3, 1, 2)
    rng = np.random.default_rng(seed)
    out = []
    for flip, aug in VERSIONS:
        p = (rng.uniform(.75, 1), rng.random(), rng.random(), *rng.uniform(.7, 1.3, 2)) if aug else (1, 0, 0, 1, 1)
        feats = []
        for i in range(0, len(x), CHUNK):
            xc = make_version(x[i:i + CHUNK].float() / 255, flip, *p)
            with torch.autocast("cuda", dtype=torch.float16):
                feats.append(backbone((xc - MEAN) / STD).float())
        out.append(torch.cat(feats).half().cpu().numpy())
    return np.stack(out)                                      # (4, T, 2048) float16

z = extract(vm.video[0], seed=1000)
z18 = np.load(SRC_DIR / vm.feat[0])["f"]
print("ResNet50:", z.shape, "| ResNet18:", z18.shape, "| hữu hạn:", bool(np.isfinite(z).all()))
assert z.shape[:2] == z18.shape[:2], "Số khung/phiên bản không khớp với bản ResNet18."

ResNet50: (4, 68, 2048) | ResNet18: (4, 68, 512) | hữu hạn: True


In [4]:
#cell 4: trích đặc trưng ResNet50 cho mọi video (bỏ qua video đã có file), cùng seed với bản ResNet18
t0, done = time.time(), 0
for i, r in vm.iterrows():
    out = OUT_DIR / r.feat
    if out.exists():
        continue
    tmp = OUT_DIR / (r.feat[:-4] + ".tmp.npz")                # ghi file tạm rồi đổi tên để file dở dang không bị tính là xong
    np.savez(tmp, f=extract(r.video, seed=1000 + i), fps=r.fps)
    tmp.replace(out)
    done += 1
    if done % 20 == 0:
        print(f"{done} video mới | vị trí {i + 1}/{len(vm)} | {time.time() - t0:.0f}s")
print(f"Xong: {done} video mới trong {time.time() - t0:.0f}s | tổng {len(vm)} video")

20 video mới | vị trí 20/215 | 26s
40 video mới | vị trí 40/215 | 53s
60 video mới | vị trí 60/215 | 80s
80 video mới | vị trí 80/215 | 107s
100 video mới | vị trí 100/215 | 167s
120 video mới | vị trí 120/215 | 205s
140 video mới | vị trí 140/215 | 253s
160 video mới | vị trí 160/215 | 302s
180 video mới | vị trí 180/215 | 361s
200 video mới | vị trí 200/215 | 431s
Xong: 215 video mới trong 467s | tổng 215 video


In [5]:
#cell 5: kiểm tra — đủ file, không NaN, cùng số khung với bản ResNet18, dung lượng
miss = [f for f in vm.feat if not (OUT_DIR / f).exists()]
print("Thiếu file:", len(miss))
assert not miss, "Chạy lại cell 4 để trích nốt các video còn thiếu."

nonfinite, shape_bad, size = 0, 0, 0
for f in vm.feat:
    z = np.load(OUT_DIR / f)["f"]
    nonfinite += int(not np.isfinite(z).all())
    shape_bad += int(z.shape[:2] != np.load(SRC_DIR / f)["f"].shape[:2])
    size += (OUT_DIR / f).stat().st_size
print(f"File có NaN/inf: {nonfinite} | file lệch số khung so với ResNet18: {shape_bad}")
print(f"Tổng dung lượng img_feats_r50: {size / 1e6:.0f} MB | ví dụ: {np.load(OUT_DIR / vm.feat[0])['f'].shape}")

Thiếu file: 0
File có NaN/inf: 0 | file lệch số khung so với ResNet18: 0
Tổng dung lượng img_feats_r50: 359 MB | ví dụ: (4, 68, 2048)
